In [ ]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/EfficientAT

import argparse
import torch
import librosa
import pandas as pd
import numpy as np
from math import ceil
from tqdm import tqdm
from torch import autocast
from contextlib import nullcontext

from models.mn.model import get_model as get_mobilenet
from models.dymn.model import get_model as get_dymn
from models.ensemble import get_ensemble_model
from models.preprocess import AugmentMelSTFT
from helpers.utils import NAME_TO_WIDTH, labels

parser = argparse.ArgumentParser(description='Example of parser. ')
parser.add_argument('--strides', nargs=4, default=[2, 2, 2, 2], type=int)
parser.add_argument('--head_type', type=str, default="mlp")
parser.add_argument('--cuda', action='store_true', default=False)

# preprocessing
parser.add_argument('--sample_rate', type=int, default=32000)
parser.add_argument('--window_size', type=int, default=800)
parser.add_argument('--hop_size', type=int, default=320)
parser.add_argument('--n_mels', type=int, default=128)

# overwrite 'model_name' by 'ensemble_model' to evaluate an ensemble
parser.add_argument('--ensemble', nargs='+', default=[])

args = parser.parse_args([])

In [ ]:
model_name = "mn04_as"
device = torch.device('cuda') if args.cuda and torch.cuda.is_available() else torch.device('cpu')
sample_rate = args.sample_rate
window_size = args.window_size
hop_size = args.hop_size
n_mels = args.n_mels

if len(args.ensemble) > 0:
    model = get_ensemble_model(args.ensemble)
else:
    if model_name.startswith("dymn"):
        model = get_dymn(width_mult=NAME_TO_WIDTH(model_name), pretrained_name=model_name,
                                strides=args.strides)
    else:
        model = get_mobilenet(width_mult=NAME_TO_WIDTH(model_name), pretrained_name=model_name,
                                strides=args.strides, head_type=args.head_type)
model.to(device)
model.eval()

# model to preprocess waveform into mel spectrograms
mel = AugmentMelSTFT(n_mels=n_mels, sr=sample_rate, win_length=window_size, hopsize=hop_size)
mel.to(device)
mel.eval()

In [ ]:
batch_size = 128
list_df = ['/run/media/fourier/Data1/Pras/DatabaseLLM/coda/longitudinal_original.csv', '/run/media/fourier/Data1/Pras/DatabaseLLM/coda/solicited_original.csv',
           '/run/media/fourier/Data1/Pras/DatabaseLLM/TBscreen_Dataset/metadata_solicited.csv', '/run/media/fourier/Data1/Pras/DatabaseLLM/TBscreen_Dataset/metadata_longitudinal.csv']
for now_csv in list_df:
    df_original = pd.read_csv(now_csv)
    paths = df_original['path_file'].tolist()

    results = []

    def batch_iterator(items, batch_size):
        for i in range(0, len(items), batch_size):
            yield items[i:i + batch_size]

    total_batches = ceil(len(paths) / batch_size)
    for batch_paths in tqdm(batch_iterator(paths, batch_size), total=total_batches):
        waveforms = []

        for p in batch_paths:
            wav, _ = librosa.load(p, sr=sample_rate, mono=True)
            wav = torch.from_numpy(wav).unsqueeze(0)  # (1, T)
            waveforms.append(wav)

        # pad to max length in batch (critical for batching)
        max_len = max(w.shape[1] for w in waveforms)
        waveforms = [
            torch.nn.functional.pad(w, (0, max_len - w.shape[1]))
            for w in waveforms
        ]

        waveform_batch = torch.stack(waveforms).to(device)  # (B, 1, T)
        waveform_batch = waveform_batch.squeeze(1)

        with torch.no_grad(), autocast(device_type=device.type) if args.cuda else nullcontext():
            spec = mel(waveform_batch)                  # (B, C, F, T)
            preds, _ = model(spec.unsqueeze(1))

        probs = torch.sigmoid(preds.float()).cpu().numpy()  # (B, num_classes)

        for path, p in zip(batch_paths, probs):
            top10 = np.argsort(p)[::-1][:20]
            cough_flag = int(47 in top10)

            results.append({
                "path_file": path,
                "cough_flag": cough_flag
            })

    df_cough_flags = pd.DataFrame(results)
    df_merged = df_original.merge(
        df_cough_flags,
        on="path_file",
        how="left"
    )
    df_merged.to_csv(now_csv + ".filtered", index=False)

In [2]:
list_df = ['/run/media/fourier/Data1/Pras/DatabaseLLM/coda/longitudinal_original.csv', '/run/media/fourier/Data1/Pras/DatabaseLLM/coda/solicited_original.csv',
           '/run/media/fourier/Data1/Pras/DatabaseLLM/TBscreen_Dataset/metadata_solicited.csv', '/run/media/fourier/Data1/Pras/DatabaseLLM/TBscreen_Dataset/metadata_longitudinal.csv']

In [30]:
df_now = pd.read_csv(list_df[1] + ".filtered")

In [41]:
from IPython.display import Audio, display

sample_data = df_now[df_now['cough_flag'] == 0].sample(n=1)['path_file'].values[0]
display(Audio(filename=sample_data))

In [4]:
df_now

,participant,filename,sound_prediction_score,sex,age,height,weight,reported_cough_dur,tb_prior,tb_prior_Pul,...,heart_rate,temperature,weight_loss,smoke_lweek,fever,night_sweats,tb_status,tb_type,path_file,cough_flag
0,CODA_TB_0002,1643799201914-recording-1.wav,0.984978,Female,48,163.0,49.9,16,Yes,Yes,...,80,36.6,Yes,No,No,Yes,1,3,/run/media/fourier/Data1/Pras/DatabaseLLM/coda...,0
1,CODA_TB_0002,1643798164476-recording-1.wav,0.981475,Female,48,163.0,49.9,16,Yes,Yes,...,80,36.6,Yes,No,No,Yes,1,3,/run/media/fourier/Data1/Pras/DatabaseLLM/coda...,0
2,CODA_TB_0002,1643798084611-recording-1.wav,0.976238,Female,48,163.0,49.9,16,Yes,Yes,...,80,36.6,Yes,No,No,Yes,1,3,/run/media/fourier/Data1/Pras/DatabaseLLM/coda...,0
3,CODA_TB_0002,1643801346641-recording-1.wav,0.980508,Female,48,163.0,49.9,16,Yes,Yes,...,80,36.6,Yes,No,No,Yes,1,3,/run/media/fourier/Data1/Pras/DatabaseLLM/coda...,0
4,CODA_TB_0003,1635236366996-recording-1.wav,0.997810,Male,51,165.0,58.0,14,Yes,No,...,100,37.0,Yes,No,No,No,0,5,/run/media/fourier/Data1/Pras/DatabaseLLM/coda...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647827,CODA_TB_1105,1645769560741-recording-1.wav,0.929447,Female,34,168.0,47.3,60,Yes,Yes,...,96,36.3,Yes,Yes,No,No,0,3,/run/media/fourier/Data1/Pras/DatabaseLLM/coda...,0
647828,CODA_TB_1106,1641287918092-recording-1.wav,0.918152,Female,63,154.0,48.0,90,No,No,...,100,36.9,No,No,No,No,1,0,/run/media/fourier/Data1/Pras/DatabaseLLM/coda...,0
647829,CODA_TB_1106,1641289828296-recording-1.wav,0.975834,Female,63,154.0,48.0,90,No,No,...,100,36.9,No,No,No,No,1,0,/run/media/fourier/Data1/Pras/DatabaseLLM/coda...,0
647830,CODA_TB_1106,1641293091855-recording-1.wav,0.948408,Female,63,154.0,48.0,90,No,No,...,100,36.9,No,No,No,No,1,0,/run/media/fourier/Data1/Pras/DatabaseLLM/coda...,1
